<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">Sebastian Raschka</a> 所著《<a href="https://mng.bz/lZ5B">从零开始构建推理模型</a>》一书的补充代码<br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px" alt="《从零开始构建推理模型》封面"></a>
</td>
</tr>
</table>

# 第六章：使用强化学习训练推理模型

本笔记本中正在使用的包：

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.16
torch version: 2.10.0
tokenizers version: 0.21.4


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F01_raschka.webp" width=600>

&nbsp;
## 6.1 大语言模型的强化学习入门

- 推理时扩展通过为每个生成的答案使用更多计算来提升推理能力
- 训练时扩展通过在训练过程中使用额外计算来提升推理能力，这正是本章的重点

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F02_raschka.webp" width=600>

- 推理时缩放与训练时缩放可以（/应该）结合使用，例如，在基于强化学习的推理训练后应用推理时技术
- 在实践中，针对大语言模型的强化学习通常作为预训练模型之后或指令微调之后的后训练阶段应用

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F03_raschka.webp" width=600>

- 预训练通过下一词预测构建通用知识，而强化学习则通过优化序列级目标（如答案正确性或偏好）来优化模型行为
- 面向大语言模型的强化学习包括推理训练和偏好调优，但推理导向的强化学习也可直接应用于预训练基础模型，如DeepSeek-R1所示
- 直接在基础模型上训练推理能力会产生较弱但仍具能力的模型（但这为理解推理阶段的贡献提供了更简单的研究场景）

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F04_raschka.webp" width=600>

&nbsp;
### 6.1.1 最初的强化学习与人类反馈流程 (RLHF)

- RLHF 于 2022 年在 InstructGPT 研究中被提出，它利用人类偏好标签来训练大语言模型（这是将 GPT-3 转变为初代 ChatGPT 的关键步骤）
- 与优化下一 token 预测的预训练和监督微调不同，RLHF 基于模型响应的人类偏好标签来优化模型

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F05_raschka.webp" width=600>

&nbsp;
### 6.1.2 从人类反馈到可验证奖励（RLVR）

- RLHF 需要训练一个单独的奖励模型，这通常是一个大型且昂贵的大语言模型
- RLVR 用自动可验证的确定性奖励取代了学习得到的奖励模型

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F06_raschka.webp" width=600>

- RLVR的普及在很大程度上得益于DeepSeek-R1在2025年的成功，该模型在不依赖人类偏好数据或学习型奖励模型的情况下展现了强大的推理能力
- DeepSeek-R1通过自动可验证的奖励来训练推理行为，例如数学问题的正确性检查以及编程任务的代码编译或执行验证
- 虽然本书聚焦于基于数学的验证方法，但其核心思想与代码验证相似：奖励通过二元成功信号自动计算得出

&nbsp;
## 6.2 使用GRPO进行可验证奖励的强化学习演练

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F07_raschka.webp" width=600>

- 现在，在介绍了整体框架并了解了强化学习如何融入大语言模型的开发周期后，我们将实现RLVR来训练一个推理模型（类似于DeepSeek-R1-Zero，但规模要小得多，因为一次可比的运行将耗费数十万美元的GPU成本）
- 大语言模型的强化学习使用一种称为策略梯度的算法来更新我们想要训练的大语言模型（在强化学习语境中称为"策略"）
- 用于RLHF的一种流行策略梯度算法是近端策略优化（PPO）；我们也可以在RLVR中使用相同的算法
- 然而，DeepSeek团队在训练DeepSeek-R1推理模型时使用了一种更简单的算法，即群组相对策略优化（GRPO）（首次在DeepSeekMath中使用）
- GRPO对资源更友好，因为在PPO中我们需要另一个大语言模型来计算价值函数；而在GRPO中则不需要，因为它从一组采样响应的相对比较中获取学习信号
- 感兴趣的读者可以在我的文章[大语言模型推理的强化学习现状](https://magazine.sebastianraschka.com/p/the-state-of-llm-reasoning-model-training)中找到PPO和GRPO更详细的对比分析
- 在本章中，我们将使用GRPO实现RLVR
- 此外，下一章将介绍对GRPO的额外改进，以提升训练稳定性和最终的建模性能

### 6.2.1 通过厨师类比理解GRPO的高层直觉

- 由于GRPO乍看可能较为复杂，我希望以一个“厨师与烹饪”的类比作为本节的开篇，通过这个类比进行宏观概述，以介绍相关术语并提供一些直觉理解。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F08_raschka.webp" width=600>

- 在大多数针对大语言模型的强化学习场景中，rollout 和 completion 是可以互换使用的术语

### 6.2.2 高级 GRPO 流程

- 在以下章节中实施GRPO的技术路线图：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F09_raschka.webp" width=600>

&nbsp;
## 6.3 加载预训练模型

本章中的代码与前面章节中的代码相同

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F10_raschka.webp" width=600 alt="第六章图10 raschka.webp">

In [2]:
import torch

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
     load_model_and_tokenizer
)

device = get_device()
device = torch.device("cpu")

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

Using Apple Silicon GPU (MPS)
✓ qwen3/qwen3-0.6B-base.pth already up-to-date


In [3]:
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache
)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

torch.manual_seed(0)
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9
)

 \boxed{58}

&nbsp;
## 6.4 加载 MATH 训练子集

- We use a non-overlapping training subset extracted from the original MATH dataset, which explicitly excludes the MATH-500 examples used in earlier chapters for model evaluation (for more information on the dataset preparation method, please refer to https://github.com/rasbt/math_full_minus_math500).

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F11_raschka.webp" width=600>

- 以下 `load_math_train` 函数与第 3 章中的 [load_math500_test](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/ch03.py#L422) 函数类似，除了我们指定了不同的文件路径。

In [4]:
import json
import requests
from pathlib import Path

def load_math_train(local_path="math_train.json", save_copy=True):
    local_path = Path(local_path)

    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "math_full_minus_math500/refs/heads/main/"
        "math_full_minus_math500.json"
    )
    backup_url = (
        "https://f001.backblazeb2.com/file/reasoning-from-scratch/"
        "MATH/math_full_minus_math500.json"
    )

    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
        except requests.RequestException:
            print("Using backup URL.")
            r = requests.get(backup_url, timeout=30)
            r.raise_for_status()

        data = r.json()

        if save_copy:
            with local_path.open("w", encoding="utf-8") as f:
                json.dump(data, f, indent=2)

    return data

In [5]:
math_train = load_math_train()

print("Dataset size:", len(math_train))

Dataset size: 12000


In [6]:
from pprint import pprint

pprint(math_train[4])

{'answer': '6',
 'level': 'Level 3',
 'problem': 'Sam is hired for a 20-day period. On days that he works, he earns '
            '$\\$$60. For each day that he does not work, $\\$$30 is '
            'subtracted from his earnings. At the end of the 20-day period, he '
            'received $\\$$660. How many days did he not work?',
 'solution': 'Call $x$ the number of days Sam works and $y$ the number of days '
             'he does not. We can set up the following system of equations to '
             'represent the given information: \\begin{align*}\n'
             'x+y &= 20 \\\\\n'
             '60x - 30y &= 660 \\\\\n'
             '\\end{align*} The first equation represents the total number of '
             'days Sam works, and the second equation represents his total '
             'profit. Solving for $x$ in the first equation yields $x = 20 - '
             'y$. Substituting into the second equation gives $60(20-y) - 30y '
             '= 660$. Canceling a factor of $10$ an

- 请注意，我们只需要 `"answer"` 和 `"problem"` 字段
- 理论上，使用 `"solution"` 字段可能很诱人，但这里我们希望让模型自由探索解决方案（而不是学习特定的解法和风格）

&nbsp;
## 6.5 采样轨迹 (Rollouts)

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F12_raschka.webp" width=600>

- Rollout 是强化学习中的术语，指生成的响应

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F13_raschka.webp" width=400>

- 我们需要使用 `@torch.no_grad`，因为我们不希望构建计算图并通过它进行反向传播，但 `@torch.inference_mode` 无法使用（它功能过多）并会导致

> RuntimeError: 推理模式下的张量无法为反向传播保存。请不要在 autograd 跟踪的计算中使用在推理模式下创建的张量。要解决此问题，可以克隆张量以获得普通张量并在 autograd 中使用，或者使用 `torch.no_grad()` 代替 `torch.inference_mode()`。

In [7]:
from reasoning_from_scratch.qwen3 import KVCache
from reasoning_from_scratch.ch04 import top_p_filter


@torch.no_grad()
def sample_response(
    model,
    tokenizer,
    prompt,
    device,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        device=device
        )

    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    logits = model(input_ids.unsqueeze(0), cache=cache)[:, -1]

    generated = []
    for _ in range(max_new_tokens):
        if temperature and temperature != 1.0:
            logits = logits / temperature

        probas = torch.softmax(logits, dim=-1)
        probas = top_p_filter(probas, top_p)
        next_token = torch.multinomial(
            probas.cpu(), num_samples=1
        ).to(device)

        token_id = next_token.item()
        generated.append(token_id)

        if (
            tokenizer.eos_token_id is not None
            and token_id == tokenizer.eos_token_id
        ):
            break
        logits = model(next_token, cache=cache)[:, -1]

    full_token_ids = torch.cat(
        [input_ids,
         torch.tensor(generated, device=device, dtype=input_ids.dtype),]
    )
    return full_token_ids, input_ids.numel(), tokenizer.decode(generated)

- 这里并无新意
- 上述代码只是我们先前开发内容的精简版本；它直接将第2章的[generate_text_basic_stream_cache](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/ch02.py#L57)函数与第4章的温度采样和top-p采样相结合
- 我们不再逐个生成token，而是将token收集到张量中，因为无需实时打印生成的token

In [8]:
torch.manual_seed(0)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

 \boxed{58}<|endoftext|>


In [9]:
torch.manual_seed(5)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

 Let's solve the problem step by step.

**Given:**
\[
\text{Half the value of } 3x - 9 \text{ is } x + 37.
\]

**Step 1: Translate the statement into an equation.**
\[
\frac{1}{2} (3x - 9) = x + 37
\]

**Step 2: Eliminate the fraction by multiplying both sides by 2.**
\[
3x - 9 = 2(x + 37)
\]

**Step 3: Distribute the 2 on the right side.**
\[
3x - 9 = 2x + 74
\]

**Step 4: Subtract \(2x\) from both sides to get the \(x\)-terms on one side.**
\[
3x - 2x - 9 = 74
\]
\[
x - 9 = 74
\]

**Step 5: Add 9 to both sides to solve for \(x\).**
\[
x = 74 + 9
\]
\[
x = 83
\]

**Final Answer:**
\[
\boxed{83}
\]<|endoftext|>


- 在实践中，我们会多次调用 `sample_response` 来生成轨迹
- 为保持 GRPO 演示流程简洁并与图6.13保持一致，我们假设模型生成了以下四个回复：

In [10]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

&nbsp;
## 6.6 计算奖励

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F14_raschka.webp" width=400>

- 奖励机制仅为正确性奖励，与第三章类似
- 但存在隐式格式奖励：仅当最终答案采用 `\boxed{}` 格式时（通过 `fallback=None` 实现）才会给予 1.0 奖励

In [11]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate, grade_answer
)

def reward_rlvr(answer_text, ground_truth):
    extracted = extract_final_candidate(
        answer_text, fallback=None  # Require \boxed{}
    )
    if not extracted:
        return 0.0
    correct = grade_answer(extracted, ground_truth)
    return float(correct)

In [12]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]
rollout_rewards = []

for answer in rollouts:
    reward = reward_rlvr(answer_text=answer, ground_truth="83")
    print(f"Answer: {answer!r}")
    print(f"Reward: {reward}\n")
    rollout_rewards.append(reward)

Answer: '\\boxed{83}'
Reward: 1.0

Answer: 'The correct answer is \\boxed{83}'
Reward: 1.0

Answer: 'The final answer is 83'
Reward: 0.0

Answer: 'We get \\boxed{38}'
Reward: 0.0



- 注意：DeepSeek-R1 团队曾尝试在训练模型时使用过程奖励模型对中间解题步骤进行评分
- 然而这些尝试并未成功，研究者最终得出结论：仅基于最终答案正确性进行奖励训练（不设置中间奖励）效果更佳

&nbsp;
## 6.7 通过优势函数从轨迹中准备学习信号

- GRPO 中的“GR”（群体相对）指的是 GRPO 为每个提示生成多个答案（rollouts），并通过相互比较来构建学习信号

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F15_raschka.webp" width=400>

- 公式非常简单：

$$\text{advantages}_i = \frac{r_i - \mu_r}{\sigma_r + \epsilon}$$

- 这里，$r_i$ 表示第 $i$ 次轨迹的奖励，$\mu_r$ 是该组轨迹的平均奖励，$\sigma_r$ 是对应的标准差，$\epsilon$ 是为数值稳定性而添加的一个小常数，以避免零除错误。

In [13]:
rewards = torch.tensor(rollout_rewards, device=device)
print(rewards)

tensor([1., 1., 0., 0.])


In [14]:
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

tensor([ 0.8659,  0.8659, -0.8659, -0.8659])


- 注意，如果一个组内的所有奖励都相同，例如全为0或全为1，那么对于所有$i$次rollout，$r_i - \mu_r = 0$
- 这意味着如果所有答案都正确或所有答案都错误，模型将不会更新

&nbsp;
## 6.8 使用序列对数概率对轨迹进行评分

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F16_raschka.webp" width=400 alt="图6.16">

- 在前一章中，我们实现了一个 `avg_logprob_answer` 函数，用于计算答案词元的逐词元对数概率
- 这些平均后的对数概率通常也被称为词级对数概率，常用于对大语言模型的答案进行评分
- 这种平均方式在评分时更受青睐，因为它提供了长度归一化
- 数学上可以表示为 $\frac{1}{T} \sum_{t=1}^{T} \log p_W(y_t \mid y_{<t}, x)$
- 其中，$y_1, ..., y_T$ 表示长度为 $T$ 的生成响应中的词元，$y_{<t}$ 表示所有先前生成的词元，$x$ 是输入提示，$W$ 表示模型的权重参数
- 该表达式在数学上与前一章使用的公式完全相同；我们只是将 $x$ 替换为 $y$，以便清晰地区分生成的输出词元和输入提示
- 供参考，该函数已复制如下

In [15]:
## Chapter 5

@torch.inference_mode()
def avg_logprob_answer(model, tokenizer, prompt, answer, device="cpu"):

    # Encode prompt and answer tokens separately to get the prompt length later
    prompt_ids = tokenizer.encode(prompt)
    answer_ids = tokenizer.encode(answer)
    full_ids = torch.tensor(prompt_ids + answer_ids, device=device)

    # Same as in calc_next_token_logprobas before
    logits = model(full_ids.unsqueeze(0)).squeeze(0)
    logprobs = torch.log_softmax(logits, dim=-1)

    # Index range for positions corresponding to answer tokens
    start = len(prompt_ids) - 1
    end = full_ids.shape[0] - 1

    # Same as before, except for using start and end
    t_idx = torch.arange(start, end, device=device)
    next_tokens = full_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # Average over the answer token scores
    return torch.mean(next_token_logps).item()

In [16]:
avg_logprob_val = avg_logprob_answer(
                   model, tokenizer, 
                   prompt=prompt,
                   answer=answer_text,
                   device=device) 
print(avg_logprob_val)

-0.061279296875


- 然而，GRPO 使用序列级别的对数概率，而非上述经过长度归一化的 token 级别平均值
- Token 级别的平均值在评分时很有用，因为它们使得不同长度的输出具有可比性
- 在 GRPO 中，每个 rollout 会为整个序列获得一个奖励和一个优势值，为了正确缩放梯度，对数概率必须反映完整序列的可能性，这通过累加 token 级别的对数概率获得
- 否则，对数概率的平均会隐式地按序列长度重新缩放学习信号，并扭曲策略更新，尤其对于较长的 rollout

- 我们可以通过去除平均化操作，并将 `torch.mean(next_token_logps)` 替换为 `torch.sum(next_token_logps)`，将其转换为序列级对数概率
- 回顾来看，我们也可以将平均结果乘以答案词元数量，以获得未平均化的值

In [17]:
sequence_logprob_val = avg_logprob_val * (len(tokenizer.encode(answer_text)))
print(sequence_logprob_val)

-16.239013671875


- 这些序列级对数概率随序列长度 T 线性增长
- 这意味着更长的回答总是会获得更负的对数概率
- 这反过来促使模型在两个同样优秀的答案中，倾向于选择更短的那个（因为成本更低）
- 累计对数概率会鼓励模型更早停止生成

- 因此，如上所述，我们可以将上述函数中的 `torch.mean` 替换为 `torch.sum`
- 然而，由于我们在前一章中使用了 `@torch.inference_mode()` 装饰器以推理模式运行该函数，我们仍需重新定义它，因为我们希望 PyTorch 能够跟踪并计算梯度
- 此外，由于我们从第 6.5 节获得了 `sample_response`，并返回了 `token_ids` 和 `prompt_len`，我们可以简化 `avg_logprob_answer` 函数，移除其中的编码和 `full_ids` 计算部分

In [18]:
def sequence_logprob_draft(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)

    # Positions whose next-token probabilities we want
    # These correspond to predicting token_ids[t + 1] from position t
    start = prompt_len - 1
    end = token_ids.shape[0] - 1

    t_idx = torch.arange(start, end, device=token_ids.device)
    next_tokens = token_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # Sum log-probabilities over the answer tokens
    return torch.sum(next_token_logps)

print(sequence_logprob_draft(model, token_ids, prompt_len))

tensor(-16.2998, grad_fn=<SumBackward0>)


- 注意，我们在 `torch.sum(next_token_logps)` 中没有使用 `.item()`，这样 PyTorch 会返回一个张量（而非 Python 浮点数），这对梯度计算至关重要
- 如我们所见，结果值（-16.2998）与之前通过答案 token 数量对 `avg_logprob_val` 进行缩放后得到的值（-16.2390）几乎一致；细微差异可归因于浮点数舍入行为

- 下面我们将使用 `torch.gather` 重写该函数，这在 PyTorch 中更符合习惯用法，并且针对 GPU 进行了更好的优化
- 然而，这两个函数在数学上是等价的

In [19]:
def sequence_logprob(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)
    selected = logprobs[:-1].gather(
        1, token_ids[1:].unsqueeze(-1)
    ).squeeze(-1)
    return torch.sum(selected[prompt_len - 1:])

print(sequence_logprob(model, token_ids, prompt_len))

tensor(-16.2998, grad_fn=<SumBackward0>)


In [20]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

rollout_logps = []

for text in rollouts:
    token_ids = tokenizer.encode(prompt + " " + text)
    logprob = sequence_logprob(
        model=model,
        token_ids=torch.tensor(token_ids, device=device),
        prompt_len=prompt_len,
    )

    print(f"Answer:  {text}")
    print(f"Logprob: {logprob.item():.4f}\n")

    rollout_logps.append(logprob)

Answer:  \boxed{83}
Logprob: -7.9243

Answer:  The correct answer is \boxed{83}
Logprob: -20.1546

Answer:  The final answer is 83
Logprob: -16.6130

Answer:  We get \boxed{38}
Logprob: -23.3677



- 这里的趋势是，更简短精炼的答案会获得更高（负值更小）的序列级对数概率
- 而唯一一个包含错误值（38而非83）的答案获得了最低分数
- 总体而言，对数概率的累加结果倾向于简洁且正确的输出

&nbsp;
## 6.9 通过GRPO损失从优势到策略更新

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F17_raschka.webp" width=400>

In [21]:
logps = torch.stack(rollout_logps)
print(logps)

tensor([ -7.9243, -20.1546, -16.6130, -23.3677], grad_fn=<StackBackward0>)


In [22]:
pg_loss = -(advantages.detach() * logps).mean()
print(pg_loss)

tensor(-2.5764, grad_fn=<NegBackward0>)


- 我们需要使用 `.detach()` 是因为希望将 `advantages` 视为固定的学习信号；这样能确保我们只通过 logprobs 进行反向传播
- 我们需要负号是因为 PyTorch 优化器默认进行最小化，而这里我们希望最大化对数概率加权的优势值

- 在数学符号表示中，策略梯度损失可写作如下形式：

$$\mathcal{L}_{\mathrm{PG}}
= -\frac{1}{N} \sum_{i=1}^{N} A_i \sum_{t=1}^{T_i} \log p_W\!\left( y_t^{(i)} \mid y_{<t}^{(i)}, x^{(i)} \right)$$

- $N$ 表示批次中的轨迹数量
- $y_1^{(i)}, ..., y_{T_i}^{(i)}$ 是第 $i$ 条长度为 $T_i$ 的生成响应的 token 序列
- $y_{<t}^{(i)}$ 表示该响应中此前已生成的所有 token
- $x^{(i)}$ 是第 $i$ 条轨迹对应的输入提示词
- $p_W$ 表示模型的策略，即由权重 $W$ 参数化的下一个 token 的概率分布
- $A_i$ 是分配给第 $i$ 条完整轨迹的优势值
- 内层求和计算单条轨迹的序列级对数概率
- 外层平均计算跨轨迹的优势加权对数概率

&nbsp;
## 6.10 在GRPO步骤中整合所有内容

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F18_raschka.webp" width=400>

In [23]:
def compute_grpo_loss(
    model,
    tokenizer,
    example,
    device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
):
    assert num_rollouts >= 2
    roll_logps, roll_rewards, samples = [], [], []
    prompt = render_prompt(example["problem"])

    was_training = model.training
    model.eval()

    for _ in range(num_rollouts):
        # Stage 1: generate rollouts
        token_ids, prompt_len, text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        # Stage 2: compute rewards
        reward = reward_rlvr(text, example["answer"])
        
        # Stage 4: compute logprobs
        logp = sequence_logprob(model, token_ids, prompt_len)

        roll_logps.append(logp)
        roll_rewards.append(reward)
        samples.append(
            {
                "text": text,
                "reward": reward,
                "gen_len": token_ids.numel() - prompt_len,
            }
        )

    if was_training:
        model.train()

    # Stage 2: collect all rewards
    rewards = torch.tensor(roll_rewards, device=device)

    # Stage 3: compute advantages
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    # Stage 4: collect all logprobs
    logps = torch.stack(roll_logps)

    # Stage 5: compute policy gradient loss
    pg_loss = -(advantages.detach() * logps).mean()
    loss = pg_loss  # In the next chapter we add a KL term here

    return {
        "loss": loss.item(),
        "pg_loss": pg_loss.item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "samples": samples,
        "loss_tensor": loss,
    }

- 代码注释中的阶段对应于 GRPO 图示中的各个阶段
- 请注意，在阶段 1 之后，代码注释中使用的是阶段 2 和 4（而非 3 和 4），因为这样能实现更简洁的代码结构（从而避免实现多重循环）

In [24]:
torch.manual_seed(123)

stats = compute_grpo_loss(
    model=model,
    tokenizer=tokenizer,
    example=math_train[4],
    device=device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9
)

pprint(stats)

{'advantages': [0.0, 0.0],
 'loss': -0.0,
 'loss_tensor': tensor(-0., grad_fn=<NegBackward0>),
 'pg_loss': -0.0,
 'rewards': [0.0, 0.0],
 'samples': [{'gen_len': 4, 'reward': 0.0, 'text': ' 14<|endoftext|>'},
             {'gen_len': 256,
              'reward': 0.0,
              'text': ' 4\n'
                      '\n'
                      "To solve the problem, let's break it down step by "
                      'step:\n'
                      '\n'
                      '1. **Define Variables:**\n'
                      '   - Let \\( x \\) be the number of days Sam works.\n'
                      '   - Then, the number of days he does not work is \\( '
                      '20 - x \\).\n'
                      '\n'
                      '2. **Set Up the Earnings Equation:**\n'
                      '   - For each day he works, he earns \\$60.\n'
                      '   - For each day he does not work, he loses \\$30.\n'
                      '   - His total earnings are \\$660.

&nbsp;
## 6.11 实现GRPO训练循环

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F19_raschka.webp" width=600>

- 由于资源需求已经很高，我们跳过了批处理

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F20_raschka.webp" width=400>

In [25]:
import time

def train_rlvr_grpo(
    model,
    tokenizer,
    math_data,
    device,
    steps=None,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=50,
    checkpoint_dir=".",
    csv_log_path=None,

):
    if steps is None:
        steps = len(math_data)

    # Stage 1: initialize optimizer
    # (the model was already initialized outside the function)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    current_step = 0
    if csv_log_path is None:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        csv_log_path = f"train_rlvr_grpo_metrics_{timestamp}.csv"
    csv_log_path = Path(csv_log_path)

    try:
        # Stage 2: Iterate over training steps
        for step in range(steps):

            # Stage 3: Reset loss gradient
            # (it's best practice to do this at the beginning of each step)
            optimizer.zero_grad()

            current_step = step + 1
            example = math_data[step % len(math_data)]

            # Stage 4: calculate GRPO loss
            stats = compute_grpo_loss(
                model=model,
                tokenizer=tokenizer,
                example=example,
                device=device,
                num_rollouts=num_rollouts,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )

            # Stage 5: Backward pass to calculate loss gradients
            stats["loss_tensor"].backward()

            # Clip large gradients to improve training stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            # Stage 6: Update model weights using loss gradients
            optimizer.step()

            # Stage 7: Collect rewards, response lengths, and losses
            reward_avg = torch.tensor(stats["rewards"]).mean().item()
            step_tokens = sum(
                sample["gen_len"] for sample in stats["samples"]
            )
            avg_response_len = (
                step_tokens / len(stats["samples"]) 
                if stats["samples"] else 0.0
            )
            append_csv_metrics(
                csv_log_path, current_step, steps, stats["loss"],
                reward_avg, avg_response_len,
            )

            # Print step metrics
            print(
                f"[Step {current_step}/{steps}] "
                f"loss={stats['loss']:.4f} "
                f"reward_avg={reward_avg:.3f} "
                f"avg_resp_len={avg_response_len:.1f}"
            )

            # Sample outputs (every 10 steps) to check if model
            # generates coherent text
            if current_step % 10 == 0:
                print(f"[Step {current_step}] sample outputs")
                for i, sample in enumerate(stats["samples"][:3]):
                    text = sample["text"].replace("\n", "\\n")
                    print(
                        f"  {i+1}) reward={sample['reward']:.3f} "
                        f"len={sample['gen_len']}: {text}"
                    )
                print()

            # Stage 8: Save model checkpoint
            if checkpoint_every and current_step % checkpoint_every == 0:
                ckpt_path = save_checkpoint(
                    model=model,
                    checkpoint_dir=checkpoint_dir,
                    step=current_step,
                )
                print(f"Saved checkpoint to {ckpt_path}")

    # Save a model checkpoint if we interrupt the training early
    except KeyboardInterrupt:
        ckpt_path = save_checkpoint(
            model=model,
            checkpoint_dir=checkpoint_dir,
            step=max(1, current_step),
            suffix="interrupt",
        )
        print(f"\nKeyboardInterrupt. Saved checkpoint to {ckpt_path}")
        return model

    return model


def save_checkpoint(model, checkpoint_dir, step, suffix=""):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    suffix = f"-{suffix}" if suffix else ""
    ckpt_path = (
        checkpoint_dir /
        f"qwen3-0.6B-rlvr-grpo-step{step:05d}{suffix}.pth"
    )
    torch.save(model.state_dict(), ckpt_path)
    return ckpt_path


def append_csv_metrics(
    csv_log_path,
    step_idx,
    total_steps,
    loss,
    reward_avg,
    avg_response_len,
):
    if not csv_log_path.exists():
        csv_log_path.write_text(
            "step,total_steps,loss,reward_avg,avg_response_len\n",
            encoding="utf-8",
        )
    with csv_log_path.open("a", encoding="utf-8") as f:
        f.write(
            f"{step_idx},{total_steps},{loss:.6f},{reward_avg:.6f},"
            f"{avg_response_len:.6f}\n"
        )

- 除了阶段4的GRPO损失计算外，其余部分均属于深度神经网络（包括大语言模型）训练时的标准流程
- `append_csv_metrics`函数将结果记录至CSV文件以便存档（同时用于第7章的结果可视化）
- 关于使用PyTorch训练神经网络的通用介绍，请参阅我的文章《一小时掌握PyTorch：从张量到多GPU神经网络训练》第3-8节（[链接](https://sebastianraschka.com/teaching/pytorch-1h/)）

In [26]:
device = get_device()
model.to(device)

torch.manual_seed(0)

train_rlvr_grpo(
    model=model,
    tokenizer=tokenizer,
    math_data=math_train,
    device=device,
    steps=50,
    num_rollouts=4,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=5,
    checkpoint_dir=".",
    csv_log_path="train_rlvr_grpo_metrics.csv",
)

Using Apple Silicon GPU (MPS)
[Step 1/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=88.0
[Step 2/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=7.5
[Step 3/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=6.5
[Step 4/50] loss=0.0909 reward_avg=0.250 avg_resp_len=6.5
[Step 5/50] loss=1.1001 reward_avg=0.500 avg_resp_len=300.5
Saved checkpoint to qwen3-0.6B-rlvr-grpo-step00005.pth

KeyboardInterrupt. Saved checkpoint to qwen3-0.6B-rlvr-grpo-step00006-interrupt.pth


Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

- 如果运行上述代码时遇到内存相关问题，可以减少rollout次数（例如 `num_rollouts=2`）和每次rollout生成的token数量（例如 `max_new_tokens=128`）
- 但要获得相对良好的模型效果，至少需要 `num_rollouts=8` 和 `max_new_tokens=512`
- 若当前硬件无法运行，无需担心，下一节将展示如何下载预训练的checkpoint

- 请注意，无论采用哪种方式，代码运行速度都会非常缓慢，因为GRPO是一个资源密集型过程
- 您可以随时中断运行，程序会在`checkpoints`文件夹中保存最新的模型检查点
- 若您对使用云端GPU感兴趣，请参阅[GPU云资源](../../ch02/02_setup-tips/gpu-instructions.md)文档获取推荐方案

- 请注意，此代码不支持批量训练
- 这是刻意为之的设计选择，旨在保持代码简洁易读，且因为采样多个（可能很长的）轨迹本身已非常消耗资源
- 但若你拥有多GPU资源，可使用本代码的可选版本（支持批量处理与多GPU加速），该版本位于补充材料目录 [../02_rlvr_grpo_scripts_intro](../02_rlvr_grpo_scripts_intro)，能显著提升模型训练速度

&nbsp;
## 6.12 加载和评估保存的模型检查点

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F21_raschka.webp" width=600>

- 可以使用第2章介绍的 `model.load_state_dict(torch.load(model_path))` 方法加载已保存的检查点，其中 `model_path` 指向检查点的 ".pth" 文件
- 这些检查点文件也与第3章的模型评估工具兼容
- 为方便起见，您可以使用第3章附赠材料中提供的评估脚本：

```python
uv run ../../ch03/02_math500-verifier-scripts/evaluate_math500.py \
--dataset_size 500 \
--which_model base \
--checkpoint_path checkpoints/qwen3-0.6B-rlvr-grpo-step00050.pth
```

- 如果您因耗时过长而不愿在本地运行 GRPO 训练，也可以下载我上传至 [rasbt/qwen3-from-scratch-grpo-checkpoints/tree/main/grpo_original_no_kl](https://huggingface.co/rasbt/qwen3-from-scratch-grpo-checkpoints/tree/main/grpo_original_no_kl) 的检查点（点击需要下载的检查点文件，然后点击[下载](https://huggingface.co/rasbt/qwen3-from-scratch-grpo-checkpoints/resolve/main/grpo_original_no_kl/qwen3-0.6B-rlvr-grpo-step00050.pth?download=true)按钮）
- 为方便起见，您也可以直接使用 Python 下载检查点

In [27]:
from reasoning_from_scratch.qwen3 import download_qwen3_grpo_checkpoints

download_qwen3_grpo_checkpoints(grpo_type="no_kl", step="00050")

✓ qwen3-0.6B-rlvr-grpo-step00050.pth already up-to-date


|      | 方法                                 | 步骤 | 最大令牌数 | 回滚次数 | MATH-500准确率 | 平均令牌数 |
| ---- | -------------------------------------- | ---- | ---------- | ------------ | ------------ | --------------- |
| 1    | 基础模型（第3章）                       | -    |            |              | 15.2%        | 78.85           |
| 2    | 推理增强（第3章）                  | -    |            |              | 48.2%        | 1369.79         |
| 3    | 原始GRPO但无KL（本章） | 50   | 512        | 8            | 47.4%        | 586.11          |

- 根据上表，我们发现仅经过50步训练后，从基础模型（第1行）初始化的训练模型（第3行）已几乎达到原始推理变体（第2行）的性能水平
- 需注意，延长训练时间未必能提升模型效果，甚至可能导致性能下降，因为GRPO算法可能相对不稳定；下一章将介绍改进GRPO算法的额外技巧

&nbsp;
## 6.13 总结

- 本节无代码